In [10]:
import pandas as pd
import os
import glob
from shutil import copytree
import numpy as np

In [ ]:
from joblib import load
def filter(df):
    rf = load('filter2_RNAi.joblib')
    features = df.drop(['ID', 'Intensity StdDev_Ch=0', 'Intensity StdDev_Ch=1', 'Intensity StdDev_Ch=2',
                       'Intensity Mean_Ch=0', 'Intensity Median_Ch=0', 'Intensity Min_Ch=0', 'Intensity Max_Ch=0'],
                         axis = 1)
    features = features.fillna(0)
    
    # rename columns to match the training dataframe with 4 channels (ch 2 = pros, ch 3 = dapi)
    mapping = {
               'Intensity Mean_Ch=1': 'Intensity Mean_Ch=2',
               'Intensity Mean_Ch=2': 'Intensity Mean_Ch=3',
               'Intensity Median_Ch=1': 'Intensity Median_Ch=2',
               'Intensity Median_Ch=2': 'Intensity Median_Ch=3',
               'Intensity Min_Ch=1': 'Intensity Min_Ch=2',
               'Intensity Min_Ch=2': 'Intensity Min_Ch=3',
               'Intensity Max_Ch=1': 'Intensity Max_Ch=2',
               'Intensity Max_Ch=2': 'Intensity Max_Ch=3'}
    features = features.rename(columns=mapping)
    
    # Preference reducing false positives
    #p_threshold = 0.60
    #output = (rf.predict_proba(features)[:, 1] > p_threshold).astype(int)
    # default settings: predict which features indicate false positives
    output = rf.predict(features)

    return df[output.astype(bool)].copy()

In [12]:
def filter_and_save_files(original_root, new_root):
    for dirpath, dirnames, filenames in os.walk(original_root):
        for file in filenames:
            if file == 'Position.csv':
                original_file_path = os.path.join(dirpath, file)
                df = pd.read_csv(original_file_path)
                
                filtered_df = filter(df)
                
                new_file_path = original_file_path.replace(original_root, new_root)
                new_dir_path = os.path.dirname(new_file_path)
                
                if not os.path.exists(new_dir_path):
                    os.makedirs(new_dir_path)
                
                filtered_df.to_csv(new_file_path, index=False)

In [ ]:
original_root = './results'
new_root = './filtered_results'

filter_and_save_files(original_root, new_root)
